# Lab 6 — End-to-end Azure AI Search RAG evaluation

This lab evaluates the live pipeline, not prepared context: Azure AI Search semantic retrieval → a Foundry model answer → deterministic retrieval and citation checks → Foundry groundedness evaluation.

The versioned dataset and metric helpers are shared with Lab 7 so the two retrieval approaches use the same contract.

## What is measured

- **Retrieval recall@k:** required evidence groups found in the actual top-k Search results.
- **Citation coverage:** factual answer units containing at least one citation.
- **Citation validity:** citations that resolve to a supplied Search result.
- **Groundedness:** Foundry `builtin.groundedness` over the actual retrieved context and generated answer.
- **Latency:** Search, generation, and total wall-clock milliseconds.
- **Cost:** observable model tokens and semantic requests; optional USD estimates use explicit subscription-specific rates from `.env`.

Current APIs: [Azure AI Search semantic ranking](https://learn.microsoft.com/azure/search/semantic-how-to-query-request), [Foundry Responses API](https://learn.microsoft.com/azure/foundry/agents/quickstarts/responses-api), and [Foundry cloud evaluation](https://learn.microsoft.com/azure/foundry/how-to/develop/cloud-evaluation).

In [ ]:
import json
import os
import re
import sys
import time
from importlib.metadata import version
from pathlib import Path

from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    helper = candidate / 'labs' / 'observability-and-evaluation' / 'rag_eval_utils.py'
    if helper.exists():
        repo_root = candidate
        sys.path.insert(0, str(helper.parent))
        break
else:
    raise FileNotFoundError('rag_eval_utils.py not found')

from rag_eval_utils import (
    citation_metrics,
    document_evidence_keys,
    estimate_cost_usd,
    format_search_context,
    load_cases,
    price_rates_from_env,
    primitive,
    response_token_usage,
    retrieval_recall,
)

load_dotenv(repo_root / '.env')
endpoint = os.getenv('FOUNDRY_PROJECT_ENDPOINT') or os.getenv('AZURE_AI_PROJECT_ENDPOINT')
model_deployment = os.getenv('FOUNDRY_MODEL') or os.getenv('AZURE_AI_MODEL_DEPLOYMENT_NAME')
search_endpoint = os.getenv('AZURE_SEARCH_ENDPOINT')
search_index = os.getenv('AZURE_SEARCH_INDEX') or os.getenv('LAB_SEARCH_INDEX')
raw_namespace = (
    os.getenv('WORKSHOP_RESOURCE_NAMESPACE')
    or os.getenv('WORKSHOP_TEAM_ID')
    or os.getenv('WORKSHOP_PARTICIPANT_ID')
    or ''
)
resource_namespace = re.sub(r'[^a-z0-9-]+', '-', raw_namespace.lower()).strip('-')[:32]
if not all((endpoint, model_deployment, search_endpoint, search_index, resource_namespace)):
    raise ValueError('Missing Foundry, Search, model, or WORKSHOP_RESOURCE_NAMESPACE configuration')
print({'namespace': resource_namespace, 'index': search_index, 'model': model_deployment, 'azure-ai-projects': version('azure-ai-projects')})

In [ ]:
dataset_path = repo_root / 'labs' / 'observability-and-evaluation' / 'data' / 'rag-evaluation-cases-v1.json'
dataset, search_cases = load_cases(dataset_path, 'azure_ai_search')
assert dataset['dataset_id'] == 'synthetic-grid-rag-e2e-v1'
assert len(search_cases) >= 3
print({'dataset': dataset['dataset_id'], 'cases': [row['case_id'] for row in search_cases]})

## Execute the live Search → answer path

Every retrieved page receives a stable `[S#]` label. The model must cite one or more of those labels after each factual sentence. The context later sent to the groundedness evaluator is the exact content returned here.

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient

credential = DefaultAzureCredential()
search_client = SearchClient(endpoint=search_endpoint, index_name=search_index, credential=credential)
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()
rates = price_rates_from_env()

search_rows = []
for case in search_cases:
    total_started = time.perf_counter()
    retrieval_started = time.perf_counter()
    documents = [
        dict(result)
        for result in search_client.search(
            search_text=case['query'],
            query_type='semantic',
            semantic_configuration_name='default',
            top=5,
            select=['id', 'vwi_code', 'title', 'excerpt', 'content', 'source_file', 'page_number'],
        )
    ]
    retrieval_ms = (time.perf_counter() - retrieval_started) * 1000
    if not documents:
        raise RuntimeError(f"{case['case_id']}: Search returned no documents")

    context = format_search_context(documents)
    generation_started = time.perf_counter()
    response = openai_client.responses.create(
        model=model_deployment,
        instructions=(
            'Answer only from the supplied sources. Put one or more citations such as [S1] '
            'at the end of every factual sentence. Never cite a source label that was not supplied. '
            'If the sources are insufficient, state exactly what is missing.'
        ),
        input=f"Question: {case['query']}\n\nSources:\n{context}",
        max_output_tokens=900,
    )
    generation_ms = (time.perf_counter() - generation_started) * 1000
    answer = response.output_text.strip()

    retrieved_keys = set().union(*(document_evidence_keys(doc) for doc in documents))
    recall = retrieval_recall(case['expected_evidence_groups'], retrieved_keys)
    citations = citation_metrics(answer, {f'S{i}' for i in range(1, len(documents) + 1)})
    usage = response_token_usage(response)
    cost = estimate_cost_usd(usage, rates)
    row = {
        'case_id': case['case_id'],
        'query': case['query'],
        'ground_truth': case['ground_truth'],
        'context': context,
        'response': answer,
        'retrieved_documents': len(documents),
        'retrieval_recall_at_5': recall['recall'],
        'recall_details': recall['groups'],
        'citation_coverage': citations['coverage'],
        'citation_validity': citations['validity'],
        'invalid_citations': citations['invalid_citations'],
        'retrieval_ms': round(retrieval_ms, 2),
        'generation_ms': round(generation_ms, 2),
        'total_ms': round((time.perf_counter() - total_started) * 1000, 2),
        **usage,
        **cost,
    }
    search_rows.append(row)
    print(json.dumps({key: row[key] for key in ('case_id', 'retrieval_recall_at_5', 'citation_coverage', 'citation_validity', 'retrieval_ms', 'generation_ms', 'model_input_tokens', 'model_output_tokens', 'estimated_cost_usd')}, indent=2))

assert len(search_rows) == len(search_cases)
assert all(row['context'] and row['response'] for row in search_rows)
print('PASS — every versioned case executed against live Search and the live model.')

## Evaluate groundedness in Foundry

This job submits the live `query`, actual retrieved `context`, and actual model `response`. The evaluator's own token use is reported separately because it is evaluation overhead, not application-path cost.

In [ ]:
from azure.ai.projects.models import TestingCriterionAzureAIEvaluator
from openai.types.eval_create_params import DataSourceConfigCustom

data_source_config = DataSourceConfigCustom(
    type='custom',
    item_schema={
        'type': 'object',
        'properties': {
            'case_id': {'type': 'string'},
            'query': {'type': 'string'},
            'context': {'type': 'string'},
            'response': {'type': 'string'},
            'ground_truth': {'type': 'string'},
        },
        'required': ['case_id', 'query', 'context', 'response', 'ground_truth'],
    },
)
groundedness_criterion = TestingCriterionAzureAIEvaluator(
    type='azure_ai_evaluator',
    name='groundedness',
    evaluator_name='builtin.groundedness',
    initialization_parameters={'model': model_deployment},
    data_mapping={
        'query': '{{item.query}}',
        'context': '{{item.context}}',
        'response': '{{item.response}}',
    },
)
run_suffix = str(int(time.time()))
eval_object = openai_client.evals.create(
    name=f'd2-e2e-search-{resource_namespace}-{run_suffix}',
    data_source_config=data_source_config,
    testing_criteria=[groundedness_criterion],
)
eval_run = openai_client.evals.runs.create(
    eval_id=eval_object.id,
    name=f'd2-e2e-search-run-{resource_namespace}-{run_suffix}',
    metadata={'namespace': resource_namespace, 'dataset': dataset['dataset_id'], 'target': 'azure-ai-search'},
    data_source={
        'type': 'jsonl',
        'source': {
            'type': 'file_content',
            'content': [
                {'item': {key: row[key] for key in ('case_id', 'query', 'context', 'response', 'ground_truth')}}
                for row in search_rows
            ],
        },
    },
)
print({'evaluation_id': eval_object.id, 'run_id': eval_run.id})

In [ ]:
deadline = time.monotonic() + 20 * 60
while eval_run.status not in ('completed', 'failed', 'canceled'):
    if time.monotonic() > deadline:
        raise TimeoutError('Groundedness evaluation exceeded 20 minutes')
    time.sleep(5)
    eval_run = openai_client.evals.runs.retrieve(run_id=eval_run.id, eval_id=eval_object.id)
    print('status:', eval_run.status)
if eval_run.status != 'completed':
    raise RuntimeError(f'Evaluation did not complete: {eval_run.status}')

output_items = list(openai_client.evals.runs.output_items.list(run_id=eval_run.id, eval_id=eval_object.id))
groundedness_scores = {}
for item in output_items:
    data = primitive(item)
    case_id = data['datasource_item']['case_id']
    result = next(result for result in data['results'] if result['name'] == 'groundedness')
    if result.get('status') != 'completed':
        raise RuntimeError(f'{case_id}: groundedness evaluator failed: {result}')
    groundedness_scores[case_id] = float(result['score'])
for row in search_rows:
    row['groundedness'] = groundedness_scores[row['case_id']]

evaluator_usage = [primitive(item) for item in (getattr(eval_run, 'per_model_usage', None) or [])]
print({'groundedness': groundedness_scores, 'evaluator_usage': evaluator_usage, 'report_url': getattr(eval_run, 'report_url', None)})
assert len(groundedness_scores) == len(search_rows)
print('PASS — Foundry groundedness returned one score for every live RAG case.')

## Release view and success check

The thresholds below are an explicit starting policy, not a universal standard. A failed quality gate is useful evidence: inspect the row-level answer and retrieval details before changing prompts or indexes.

In [ ]:
thresholds = {
    'retrieval_recall_at_5': 1.0,
    'citation_coverage': 0.80,
    'citation_validity': 1.0,
    'groundedness': 3.0,
}
for row in search_rows:
    row['release_gate_passed'] = all(row[metric] >= minimum for metric, minimum in thresholds.items())
    print(json.dumps({
        'case_id': row['case_id'],
        'recall@5': row['retrieval_recall_at_5'],
        'citation_coverage': row['citation_coverage'],
        'citation_validity': row['citation_validity'],
        'groundedness': row['groundedness'],
        'latency_ms': {'retrieval': row['retrieval_ms'], 'generation': row['generation_ms'], 'total': row['total_ms']},
        'usage': {'input_tokens': row['model_input_tokens'], 'output_tokens': row['model_output_tokens'], 'semantic_requests': row['semantic_requests']},
        'estimated_cost_usd': row['estimated_cost_usd'],
        'release_gate_passed': row['release_gate_passed'],
    }, indent=2))

assert all(0.0 <= row['retrieval_recall_at_5'] <= 1.0 for row in search_rows)
assert all(0.0 <= row['citation_coverage'] <= 1.0 for row in search_rows)
assert all(row['retrieval_ms'] > 0 and row['total_ms'] >= row['retrieval_ms'] for row in search_rows)
assert all(row['groundedness'] >= 1.0 for row in search_rows)
print('SUCCESS CHECK — recall, citations, groundedness, latency, and cost dimensions are populated.')

## Participant challenge

Add one representative procedure case to `data/rag-evaluation-cases-v1.json`, rerun the lab, and explain whether a failure is caused by retrieval, generation, citation behavior, or the release threshold. Do not lower a threshold without a documented reason.

In [ ]:
# TODO: add a participant-owned case and compare its row-level metrics.
# Keep the case synthetic and give each required evidence group a stable identifier.

In [ ]:
allow_cleanup = os.getenv('WORKSHOP_ALLOW_CLEANUP', 'false').lower() == 'true'
if allow_cleanup:
    openai_client.evals.delete(eval_id=eval_object.id)
    print('Deleted this namespaced evaluation and its run.')
else:
    print('Cleanup disabled so the Foundry evaluation report remains available.')